# BACKPROPAGATION FROM SCRATCH

Let's try implementing a 2-layer network (1 hidden layer) for binary classification. The architecture we are gonna follow is going to be:
Input → ReLU → Sigmoid → Loss
So let's get going now

In [1]:
# Let's start by importing the main library we will be using for this task i.e Numpy

In [2]:
import numpy as np

In [3]:
# Now let's define our loss functions and their gradients

In [4]:
def relu(Z):
    return np.maximum(0, Z)

def relu_derivative(Z):
    return (Z > 0).astype(float)

def sigmoid(Z):
    return 1 / (1 + np.exp(-Z))

def sigmoid_derivative(Z):
    s = sigmoid(Z)
    return s * (1 - s)

In [5]:
# Next come's our binary cross entropy loss function
# Since this is a binary classification task this is the best loss function for our model

In [6]:
def compute_loss(y_hat, y):
    m = y.shape[1]
    y_hat = np.clip(y_hat, 1e-9, 1 - 1e-9)
    loss = -np.mean(y * np.log(y_hat) + (1 - y) * np.log(1 - y_hat))
    return loss

In [7]:
# This step initializes our array to store weights
def initialize_params(n_x, n_h, n_y):
    np.random.seed(42)
    W1 = np.random.randn(n_h, n_x) * 0.01
    b1 = np.zeros((n_h, 1))
    W2 = np.random.randn(n_y, n_h) * 0.01
    b2 = np.zeros((n_y, 1))
    return {"W1": W1, "b1": b1, "W2": W2, "b2": b2}

In [8]:
# Let's define forward pass now

In [11]:
def forward_pass(X, params):
    W1, b1 = params["W1"], params["b1"]
    W2, b2 = params["W2"], params["b2"]

    Z1 = np.dot(W1, X) + b1
    A1 = relu(Z1)
    Z2 = np.dot(W2, A1) + b2
    A2 = sigmoid(Z2)

    cache = {"Z1": Z1, "A1": A1, "Z2": Z2, "A2": A2} # This will store our intermediate values for BackProp
    return A2, cache

In [12]:
# Now let's do the Backward Pass.
# This is the core. We compute dL/dW and dL/db for each layer, while flowing gradients backward using the chain rule.

# Layer 2 (output):
#   dL/dZ2 = A2 - y              ← special case because Binary Cross Entropy + sigmoid
#   dL/dW2 = (1/m) · dZ2 · A1ᵀ
#   dL/db2 = (1/m) · sum(dZ2)
#   dL/dA1 = W2ᵀ · dZ2

# Layer 1 (hidden):
#   dL/dZ1 = dL/dA1 · ReLU'(Z1)
#   dL/dW1 = (1/m) · dZ1 · Xᵀ
#   dL/db1 = (1/m) · sum(dZ1)

In [13]:
def backward_pass(X, y, params, cache):

    m = X.shape[1]

    W1 = params["W1"]
    W2 = params["W2"]
    A1 = cache["A1"]
    A2 = cache["A2"]
    Z1 = cache["Z1"]

    dZ2 = A2 - y
    dW2 = (1/m) * np.dot(dZ2, A1.T)
    db2 = (1/m) * np.sum(dZ2, axis=1, keepdims=True)

    dA1 = np.dot(W2.T, dZ2)

    dZ1 = dA1 * relu_derivative(Z1)
    dW1 = (1/m) * np.dot(dZ1, X.T)
    db1 = (1/m) * np.sum(dZ1, axis=1, keepdims=True)

    grads = {"dW1": dW1, "db1": db1, "dW2": dW2, "db2": db2}
    return grads

In [14]:
# Now we will update the gradients

In [15]:
def update_params(params, grads, learning_rate=0.01):
    params["W1"] -= learning_rate * grads["dW1"]
    params["b1"] -= learning_rate * grads["db1"]
    params["W2"] -= learning_rate * grads["dW2"]
    params["b2"] -= learning_rate * grads["db2"]
    return params


In [16]:
# Let's train the model now

In [17]:
def train(X, y, n_h, epochs=1000, learning_rate=0.01):

    n_x = X.shape[0]
    n_y = y.shape[0]

    params = initialize_params(n_x, n_h, n_y)
    loss_history = []

    for epoch in range(epochs):

        A2, cache = forward_pass(X, params)

        loss = compute_loss(A2, y)
        loss_history.append(loss)

        grads = backward_pass(X, y, params, cache)

        params = update_params(params, grads, learning_rate)

        if epoch % 100 == 0:
            print(f"Epoch {epoch:4d} | Loss: {loss:.4f}")

    return params, loss_history


def predict(X, params):
    A2, _ = forward_pass(X, params)
    return (A2 > 0.5).astype(int)

In [18]:
# This wraps up our Backpropagation model

In [19]:
# Let's try it once by soling the XOR truth table and seeing how well our model does on it

In [22]:
X = np.array([[0, 0, 1, 1], [0, 1, 0, 1]])
y = np.array([[0, 1, 1, 0]])

params, loss_history = train(X, y, n_h=4, epochs=2000, learning_rate=0.1)
preds = predict(X, params)
print("Predictions:", preds)
print("Reality:", y)
print("Accuracy:", np.mean(preds == y))

Epoch    0 | Loss: 0.6931
Epoch  100 | Loss: 0.6931
Epoch  200 | Loss: 0.6931
Epoch  300 | Loss: 0.6929
Epoch  400 | Loss: 0.6922
Epoch  500 | Loss: 0.6892
Epoch  600 | Loss: 0.6778
Epoch  700 | Loss: 0.6415
Epoch  800 | Loss: 0.5761
Epoch  900 | Loss: 0.5245
Epoch 1000 | Loss: 0.4990
Epoch 1100 | Loss: 0.4801
Epoch 1200 | Loss: 0.4161
Epoch 1300 | Loss: 0.3015
Epoch 1400 | Loss: 0.2089
Epoch 1500 | Loss: 0.1465
Epoch 1600 | Loss: 0.1074
Epoch 1700 | Loss: 0.0822
Epoch 1800 | Loss: 0.0649
Epoch 1900 | Loss: 0.0532
Predictions: [[0 1 1 0]]
Reality: [[0 1 1 0]]
Accuracy: 1.0


In [23]:
# We notice a 100% accuracy which shows our model works well (also to note that this accuracy just comes out to be 75% at 1000 epochs)